# Implementing a 5 qubit Quantum Fourier Transform
**Computer Laboratory II (417526) – Quantum AI – Practical 5**

Install once (Linux terminal):
```
pip install qiskit qiskit-aer matplotlib pylatexenc
```
Runs on the local Aer simulator – no IBM account or API token needed (the lab manual's `execute` / `IBMQ` / `qiskit.aqua` / `qiskit.ignis` code was removed from current Qiskit, so this notebook uses the current API).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import qiskit, qiskit_aer
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

sim = AerSimulator()
print("qiskit", qiskit.__version__, "| qiskit-aer", qiskit_aer.__version__)

def run(qc, shots=1024, backend=sim, **kw):
    # Transpile for the backend, run, and return the counts dictionary.
    t = transpile(qc, backend, **kw)
    return backend.run(t, shots=shots).result().get_counts()

## Practical 5 – 5-Qubit Quantum Fourier Transform
**Theory:** The QFT maps the computational basis to the Fourier (Hadamard-type) basis. For each qubit (from the top) it applies a Hadamard, then controlled phase rotations of π/2, π/4, π/8, … from the qubits below it; finally the qubit order is reversed with SWAP gates.
(The manual's *controlled-U1* gate is the same as Qiskit's `cp` – controlled phase.)

* Encode the value **10101** (X on q0, q2, q4) → QFT → measuring gives **random** results, because the value is now stored in the phases.
* Applying the **inverse QFT** reverses all the rotations and gives **10101** back.

In [ ]:
def qft_rotations(qc, n):
    # Recursive QFT rotations on the first n qubits
    if n == 0:
        return qc
    n -= 1
    qc.h(n)
    for qubit in range(n):
        qc.cp(np.pi / 2 ** (n - qubit), qubit, n)     # pi/2, pi/4, pi/8, ...
    return qft_rotations(qc, n)

def swap_registers(qc, n):
    for q in range(n // 2):
        qc.swap(q, n - q - 1)
    return qc

def qft(n):
    qc = QuantumCircuit(n, name='QFT')
    qft_rotations(qc, n)
    swap_registers(qc, n)
    return qc

n = 5
display(qft(n).draw(output='mpl'))

In [ ]:
# QFT of the encoded value 10101
circ = QuantumCircuit(n, n)
for q in (0, 2, 4):
    circ.x(q)                           # encode 10101
circ.compose(qft(n), inplace=True)
circ.measure(range(n), range(n))

counts = run(circ, shots=1000)
print("QFT output: ", len(counts), "different values observed (spread over all 32) ->")
print(dict(sorted(counts.items())))
plot_histogram(counts, figsize=(14, 4), title='5-qubit QFT of 10101 (random-looking)')

In [ ]:
# QFT followed by the inverse QFT -> original value comes back
circ2 = QuantumCircuit(n, n)
for q in (0, 2, 4):
    circ2.x(q)
circ2.compose(qft(n), inplace=True)
circ2.barrier()
circ2.compose(qft(n).inverse(), inplace=True)
circ2.measure(range(n), range(n))

counts2 = run(circ2, shots=1000)
print("QFT + inverse QFT:", counts2)         # expect {'10101': 1000}
display(circ2.draw(output='mpl', fold=-1))

In [ ]:
# Optional: verify our QFT against Qiskit's built-in implementation
from qiskit.quantum_info import Operator
try:
    from qiskit.circuit.library import QFTGate
    ref = QuantumCircuit(n); ref.append(QFTGate(n), range(n))
except ImportError:                       # older Qiskit
    from qiskit.circuit.library import QFT
    ref = QFT(n)
print("Matches Qiskit's library QFT:", Operator(qft(n)).equiv(Operator(ref)))